## Imports

In [1]:
# Standard Libraries
import os
import csv
import math
import copy
import pickle
import itertools
import cProfile
import pstats
from functools import reduce
from datetime import datetime
import datetime as dt
from collections import namedtuple

# Data Handling & Analysis
import numpy as np
import pandas as pd
import hydroeval as he
from sklearn.metrics import mean_squared_error

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objs as go
import seaborn as sns
from adjustText import adjust_text

# SWMM API
from swmm_api import __version__, read_rpt_file, read_inp_file, read_out_file, swmm5_run
from swmm_api.output_file import OBJECTS, VARIABLES
from swmm_api.input_file import SwmmInput, section_labels as sections
from swmm_api.input_file.sections import Outfall
from swmm_api.input_file.section_labels import TIMESERIES
from swmm_api.input_file.macros.plotting_map import plot_map, add_node_labels
from swmm_api.input_file.macros.plotting_longitudinal import plot_longitudinal


### Functions

In [2]:
def load_runoff_obs_to_df(OBS_RUNOFF_DATA_PATH):
    """ Takes OBSERVATION FILE with runoff data and makes pd Dataframe
    :param output_path: str, path name that have the runoff data
    :return: df_obs_runoff
    """
    df_obs_runoff = pd.read_csv (OBS_RUNOFF_DATA_PATH + ".csv")
    df_obs_runoff.dropna(inplace = True) ; df_obs_runoff.rename(columns={'discharge_cms': 'OBS runoff [CMS]'} , inplace=True, errors='raise')
    df_obs_runoff["date_and_time"] =  pd.to_datetime(df_obs_runoff["date_and_time"], format='%d/%m/%Y %H:%M:%S')
    df_obs_runoff = df_obs_runoff.set_index('date_and_time')
    df_obs_runoff.drop(['Unnamed: 0'], axis=1, inplace=True)


    return df_obs_runoff
    
# df_runoff.to_pickle('df_runoff.pkl')

In [3]:
def data_5min_interpolate(df_to_interpolate):
    """ Takes df column and makes 5 min interpolation
    :param df_to_interpolate: pd, df with index datetime64 type
    :return: df_interpol
    """
    df_interpol = df_to_interpolate.resample('5min').mean()
#     df_interpol= df_interpol.interpolate(method='polynomial', order=3)
    df_interpol= df_interpol.interpolate(method='linear')
    return df_interpol

In [4]:
def load_sim_to_df(sim_output_path):
    """ Takes SWMM FILE with runoff data and makes pd Dataframe
    sim_output_path: str, file name that have the swmm outflow data
    return: sim_df
    """
    sim_df = read_out_file(sim_output_path).to_frame()['system'][''][['outflow', 'rainfall']]
    sim_df.rename(columns={'outflow':'SWMM outflow [CMS]', 'rainfall':'rainfall [mm/h]'}, inplace=True, errors='raise')
    sim_df.index = pd.to_datetime(sim_df.index)
    return sim_df

In [5]:
def update_imperviousness_with_factor(subcatchment_dict, factor):
    """
    Update the imperviousness values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which imperviousness values need to be powered.

    Returns:
        dict: The updated dictionary with imperviousness values powered by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
#         initial_imperviousness = impervious_initial_values[index]
#         subcatchment.imperviousness = initial_imperviousness
        powered_imperviousness = subcatchment.imperviousness * factor
        # Update the imperviousness value for the current subcatchment with the powered imperviousness value
        subcatchment.imperviousness = powered_imperviousness


    # Return the updated dictionary
    return subcatchment_dict

def update_storage_with_factor(subareas_dict, factor):
    """
    Update the storage values of SubArea objects in a dictionary by a factor.

    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        factor (float): The factor by which storage values need to be powered.

    Returns:
        dict: The updated dictionary with storage values powered by the factor.
    """
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        # Access the impervious and pervious storage values for the current subarea
        imp_storage = subarea.storage_imperv
        perv_storage = subarea.storage_perv
        # Multiply the impervious and pervious storage values by the factor
        powered_imp_storage = imp_storage * factor
        powered_perv_storage = perv_storage * factor
        # Update the storage values for the current subarea with the powered values
        subarea.storage_imperv = powered_imp_storage
        subarea.storage_perv = powered_perv_storage

    # Return the updated dictionary
    return subareas_dict

def update_n_with_factor(subareas_dict, factor):
    """
    Update the n_imperv and n_perv values of SubArea objects in a dictionary by a factor.

    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        factor (float): The factor by which n_imperv and n_perv values need to be multiplied.

    Returns:
        dict: The updated dictionary with n_imperv and n_perv values multiplied by the factor.
    """
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        # Access the impervious and pervious n values for the current subarea
        imp_n = subarea.n_imperv
        perv_n = subarea.n_perv
        # Multiply the impervious and pervious n values by the factor
        updated_imp_n = imp_n * factor
        updated_perv_n = perv_n * factor
        # Update the n values for the current subarea with the multiplied values
        subarea.n_imperv = updated_imp_n
        subarea.n_perv = updated_perv_n

    # Return the updated dictionary
    return subareas_dict


def update_width_with_factor(subcatchment_dict, factor, width_initial_values):
    """
    Update the width values of SubCatchment objects in a dictionary by a factor.

    Args:
        subcatchment_dict (dict): A dictionary containing SubCatchment objects as values with subcatchment names as keys.
        factor (float): The factor by which width values need to be powered.
        width_initial_values (numpy.ndarray): A NumPy array of initial width values corresponding to each subcatchment.

    Returns:
        dict: The updated dictionary with width values powered by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, subcatchment) in enumerate(subcatchment_dict.items()):
        width = width_initial_values[index]  # Get the initial width value for the current subcatchment
        powered_width = width * factor
        subcatchment.width = powered_width

    # Return the updated dictionary
    return subcatchment_dict


def update_curve_num_with_factor(infiltration_dict, factor, curve_no_initial_values):
    """
    Update the curve_no values of InfiltrationCurveNumber objects in a dictionary by a factor.

    Args:
        infiltration_dict (dict): A dictionary containing InfiltrationCurveNumber objects as values with subcatchment
                                  names as keys.
        factor (float): The factor by which curve_no values need to be powered.
        curve_no_initial_values (numpy.ndarray): A NumPy array of initial curve_no values corresponding to each subcatchment.

    Returns:
        dict: The updated dictionary with curve_no values powered by the factor.
    """
    # Loop through each subcatchment in the dictionary
    for index, (subcatchment_name, infiltration_curve) in enumerate(infiltration_dict.items()):
        curve_no = curve_no_initial_values[index]  # Get the initial curve_no value for the current subcatchment
        powered_curve_no = curve_no * factor
        infiltration_curve.curve_no = powered_curve_no

    # Return the updated dictionary
    return infiltration_dict


def update_pct_zero_with_factor(subareas_dict, pct_zero_factor, pct_zero_initial_values):
    """
    Update the pct_zero values of SubArea objects in a dictionary by a factor.
    
    Args:
        subareas_dict (dict): A dictionary containing SubArea objects as values with subarea names as keys.
        pct_zero_factor (float): The factor by which pct_zero values need to be multiplied.
        pct_zero_initial_values (numpy.ndarray): A NumPy array of initial pct_zero values corresponding to each subarea.
    
    Returns:
        dict: The updated dictionary with pct_zero values multiplied by the factor.
    """
    # Loop through each subarea in the dictionary
    for index, (subarea_name, subarea) in enumerate(subareas_dict.items()):
        # Get the initial pct_zero value for the current subarea
        pct_zero = pct_zero_initial_values[index]
        # Multiply the pct_zero value by the factor
        updated_pct_zero = pct_zero * pct_zero_factor
        # Update the pct_zero value for the current subarea with the multiplied value
        subarea.pct_zero = updated_pct_zero
    
    # Return the updated dictionary
    return subareas_dict


def update_pct_routed_with_factor(subareas_dict, routed_to, precent):
    # Loop through each subarea in the dictionary
    for subarea_name, subarea in subareas_dict.items():
        subarea.route_to = routed_to
        subarea.pct_routed = precent

        # Return the updated dictionary
    return subareas_dict


In [6]:
def update_TimeSeriesData(timeseries_dict, basins_rain_df):
    """
    Update the TimeSeriesData values of SWMM inp file by using basins_rain_df

    Args:
        timeseries_dict (dict): A dictionary containing TimeSeries name as a key for each basin.
        basins_rain_df (DataFrame): DataFrame where each row represents a basin, and each column represents a timestep. The values are rain (mm) for each basin in a timestep.

    Returns:
        str: Text representation of the updated timeseries data in the specified format.
    """
    # Initialize an empty string to store the text
    
    timeseries_text = ''
    # Write the header
    timeseries_text += ";;Name                 Date          Time         Value     \n"
    timeseries_text += ";;------------ ----------------- ------------- -------------\n"

    # Iterate over each row in the DataFrame
    for index, row in basins_rain_df.iterrows():
        # Get the basin name
        basin_name = int(row['Basin_name'])

        # Write the data to the text
        timestamps = basins_rain_df.columns[1:]  # excluding the first column 'Basin_name'
        values = row.values[1:]  # excluding the first column 'Basin_name'
        dates = [ts.strftime('%m/%d/%Y') for ts in timestamps]  # convert date format
        times = [ts.strftime('%H:%M:%S') for ts in timestamps]  # convert time format
        for i in range(len(timestamps)):
            timeseries_text += f"  {date}_S{basin_name}      {dates[i]}    {times[i]}   {values[i]:.4f}\n"
        timeseries_text += ";;------------ ----------------- ------------- -------------\n"
    return timeseries_text


In [7]:
def stat(storm_df):
    """
    Format statistical data from storm_df DataFrame.
    
    Calculates various statistical parameters based on the storm data in the storm_df DataFrame.
    
    Args:
        storm_df (pandas.DataFrame): DataFrame containing storm data from SWMM and OBS.
        
    Returns:
        Tuple containing the following:
        
        - swmm_total_runoff (float): Total runoff volume in cubic meters calculated from SWMM data.
        - obs_total_runoff (float): Total runoff volume in cubic meters calculated from OBS data.
        - swmm_max_runoff (float): Runoff peak in CMS (Cubic Meters per Second) from SWMM data.
        - obs_max_runoff (float): Runoff peak in CMS (Cubic Meters per Second) from OBS data.
        - swmm_max_runoff_time (datetime): Time of the runoff peak from SWMM data.
    """
    
    # Total runoff volume [cubic meter]
    swmm_total_runoff = sum(storm_df['SWMM outflow [CMS]'])
    obs_total_runoff = sum(storm_df['OBS runoff [CMS]'])

    # Runoff peak [CMS] and time
    swmm_max_runoff = storm_df['SWMM outflow [CMS]'].max()
    swmm_max_runoff_time = storm_df[storm_df['SWMM outflow [CMS]'] == swmm_max_runoff].index[0]

    obs_max_runoff = storm_df['OBS runoff [CMS]'].max()
    obs_max_runoff_time = storm_df[storm_df['OBS runoff [CMS]'] == obs_max_runoff].index[0]

    return swmm_total_runoff, obs_total_runoff, swmm_max_runoff, obs_max_runoff, swmm_max_runoff_time


In [8]:
def observation_and_swmm_hydrograph(storm_df, date):
    """
    This function takes observation and SWMM model runoff and precipitation data and puts it in a hydrograph.
    Params:
     - storm_df: df, The DataFrame that includes the runoff and precipitation data
     - date: date of storm occurrence. Format: 'yyyy_mm_dd'
    Returns: a hydrograph plot for one subcatchment
    """
    
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.suptitle(storm_df.index.strftime('%d/%m/%Y')[0], fontsize=20)

    y_obs_runoff = storm_df['OBS runoff [CMS]']
    y_rainfall = storm_df['rainfall [mm/h]']
    y_sim_runoff = storm_df['SWMM outflow [CMS]']

    ax1 = sns.lineplot(ax=ax, data=y_obs_runoff, color='g', label='Observed Runoff')
    ax2 = ax1.twinx()
#     ax2.set_ylabel('Rainfall ($mm/hr$)')

    sns.lineplot(ax=ax2, data=y_rainfall, color='b', label='Rainfall', alpha=0.4)
    ax2.fill_between(storm_df.index, 0, y_rainfall, alpha=0.4, color='b')
#     ax2.set_ylim(ymin=y_rainfall.max() + 50, ymax=0)
    ax2.set_ylim(ymin=70, ymax=0)
    ax1.set_ylim(ymin=0, ymax=90)
    

    sns.lineplot(ax=ax, data=y_sim_runoff, color='r', label='Simulated outflow')

    ax2.set_xlabel('Time', fontsize=18)
    ax2.set_ylabel('Rainfall ($mmhr^{-1}$)', fontsize=18)
    ax1.set_ylabel('Runoff ($m^{3}s^{-1}$)', fontsize=18)

    ax2.legend().remove()

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    handles = handles1 + handles2
    labels = labels1 + labels2
    ax.legend(handles, labels, loc='center right', fontsize=15)

    # set x-axis label format to %d/%m %H
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m %H:%M'))
    ax.tick_params(axis='x', labelsize=18, rotation=45)
    ax1.tick_params(axis='y', labelsize=18) ; ax2.tick_params(axis='y', labelsize=18)
    ax1.grid()

    return fig


In [9]:
def calculate_objectives(observed, simulated):
    """
    Calculate the Nash-Sutcliffe Efficiency (NSE), Root Mean Square Deviation (RMSD), bias, Mean Absolute Deviation (MAD), and R-squared (R^2) between observed and simulated data.

    Parameters:
    observed (pandas.core.series.Series): Series containing the observed data.
    simulated (pandas.core.series.Series): Series containing the simulated data.

    Returns:
    nse (float): Nash-Sutcliffe Efficiency (NSE) value.
    rmsd (float): Root Mean Square Deviation (RMSD) value.
    bias (float): Bias value.
    mad (float): Mean Absolute Deviation (MAD) value.
    r_squared (float): R-squared (R^2) value.
    kge (float): KGE value.
    """
    if not isinstance(observed, np.ndarray):
        observed = observed.to_numpy()
    if not isinstance(simulated, np.ndarray):
        simulated = simulated.to_numpy()

    observed_mean = np.mean(observed)

    ss_diff = np.sum((observed - simulated) ** 2)
    ss_mean = np.sum((observed - observed_mean) ** 2)
    nse = 1 - (ss_diff / ss_mean)
    rmsd = np.sqrt(np.mean((observed - simulated) ** 2))
    bias = abs(np.mean(simulated - observed))
    mad = np.mean(np.abs(simulated - observed))
    kge_tuple = he.evaluator(he.kge, simulated, observed)
    kge = float(kge_tuple[0])
    # Calculate R-squared (R^2)
    ss_total = np.sum((observed - observed_mean) ** 2)
    r_squared = 1 - (ss_diff / ss_total)
#     print("mean" , ss_mean)
#     print(ss_total)
    return nse, rmsd, bias, mad, r_squared, kge


In [10]:
def compute_objective_functions(obs_hydrograph_df, cali_results_df, cali_swmm_hydrograph_df):
    # Define storm objective functions
    storm_objective_functions = ['storm_1-nse', 'storm_rmsd', 'storm_bias', 'storm_mad', 'storm_r_squared', 'storm_1-kge']
    columns = pd.MultiIndex.from_tuples([('storm objective functions', col) for col in storm_objective_functions])
    storm_objective_function_df = pd.DataFrame(columns=columns)

    # Define peak objective functions
    peak_objective_functions = ['peak_1-nse', 'peak_rmsd', 'peak_bias', 'peak_mad', 'peak_r_squared', 'peak_1-kge']
    columns = pd.MultiIndex.from_tuples([('peak objective functions', col) for col in peak_objective_functions])
    peak_objective_function_df = pd.DataFrame(columns=columns)

    # Define volume objective functions
    volume_objective_functions = ['volume_1-nse', 'volume_rmsd', 'volume_bias', 'volume_mad', 'volume_r_squared', 'volume_1-kge']
    columns = pd.MultiIndex.from_tuples([('volume objective functions', col) for col in volume_objective_functions])
    volume_objective_function_df = pd.DataFrame(columns=columns)

    # Compute storm objective functions
    obs_hydrograph_arr = obs_hydrograph_df['OBS runoff [CMS]'].values.astype(np.float64)
    for idx, row in cali_swmm_hydrograph_df.drop('Factors', level=0, axis=1).iterrows():
        simulated_hydrograph = cali_swmm_hydrograph_df.drop('Factors', level=0, axis=1).iloc[idx].astype(np.float64)
        storm_nse, storm_rmsd, storm_bias, storm_mad, storm_r_squared, storm_kge = calculate_objectives(obs_hydrograph_arr, simulated_hydrograph)
        storm_objective_function_df.loc[idx] = [(1 - storm_nse), storm_rmsd, storm_bias, storm_mad, storm_r_squared, (1 - storm_kge)]

    # Compute peak objective functions
    obs_max_runoff = obs_hydrograph_df.groupby(level=0)['OBS runoff [CMS]'].max().astype(np.float64)
    cali_swmm_peak_flow_df = cali_results_df.xs('Max Runoff', level=1, axis=1, drop_level=False).astype(np.float64)
    for idx, row in cali_swmm_hydrograph_df.drop('Factors', level=0, axis=1).iterrows():
        simulated_peak = cali_swmm_peak_flow_df.iloc[idx]
        peak_nse, peak_rmsd, peak_bias, peak_mad, peak_r_squared, peak_kge = calculate_objectives(obs_max_runoff, simulated_peak)
        peak_objective_function_df.loc[idx] = [(1 - peak_nse), peak_rmsd, peak_bias, peak_mad, peak_r_squared, (1 - peak_kge)]

    # Compute volume objective functions
    obs_storm_volume = (obs_hydrograph_df.groupby(level=0)['OBS runoff [CMS]'].sum() * 5 * 60).astype(np.float64)
    cali_swmm_total_volume_df = (cali_results_df.xs('Total Volume', level=1, axis=1, drop_level=False) * 5 * 60).astype(np.float64)
    for idx, row in cali_swmm_hydrograph_df.drop('Factors', level=0, axis=1).iterrows():
        simulated_volume = cali_swmm_total_volume_df.iloc[idx]
        volume_nse, volume_rmsd, volume_bias, volume_mad, volume_r_squared, volume_kge = calculate_objectives(obs_storm_volume, simulated_volume)
        volume_objective_function_df.loc[idx] = [(1 - volume_nse), volume_rmsd, volume_bias, volume_mad, volume_r_squared, (1 - volume_kge)]

    return storm_objective_function_df, peak_objective_function_df, volume_objective_function_df


In [11]:
# %load_ext memory_profiler
def run_calibration(MAIN_PATH, FACTOR_COMBINATION):
    global count, obs_hydrograph_df, imp_factor_l, width_factor_l, storage_factor_l, pct_routed_factor_l, evap_factor_l
    global pct_zero_factor_l, n_factor_l, cali_results_df_l, hydrograph_df_l, swmm_runoff_l
    global swmm_max_runoff_time_l, swmm_total_runoff_l, swmm_max_runoff_l

    count = 0
    obs_hydrograph_df = pd.DataFrame()
    # Define the initial values as NumPy arrays
#     width_initial_values = np.array([896, 1161, 680, 503, 545, 1222, 860, 447, 777, 889, 1348, 476, 1180, 451, 846, 715, 1067, 669, 617, 565, 1095, 1255, 1005, 394, 399, 771, 1366, 471]) # Av overland flow method
    width_initial_values = np.array([2470, 2800, 2800, 1890, 1160, 2840, 2660, 2000, 1140, 1400, 540, 2200, 4160, 1070, 1310, 1310, 1200, 1440, 1400, 1400, 1520, 1850, 1740, 2504, 980, 1900, 2140, 950]) # W=2L method     curve_no_initial_values = np.array([71, 61, 61, 61, 61, 61, 61, 61, 71, 61, 61, 71, 66, 61, 66, 71, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 71])
    curve_no_initial_values = np.array([39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39])
    pct_zero_initial_values = np.array([30, 50, 50, 50, 50, 50, 50, 50, 30, 50, 50, 30, 30, 50, 30, 50, 50, 50, 50, 50, 50, 60, 50, 50, 50, 60, 60, 30])
#     curve_no_initial_values = np.array([71, 61, 61, 61, 61,  61, 61, 61, 71, 61, 61, 71, 66, 61, 66, 71, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 61, 71])

# Call the update function with the initial pct_zero values array

    imp_factor_l = [] ; width_factor_l = [] ; storage_factor_l = [] ; pct_zero_factor_l = [] ; pct_routed_factor_l = []
    evap_factor_l = [] ; n_factor_l = [] ; cn_factor_l = []; cali_results_df_l = [] ; hydrograph_df_l = [] ; swmm_runoff_l = [] 
    swmm_max_runoff_time_l = [] ; swmm_total_runoff_l = [] ; swmm_max_runoff_l = []

    for date in os.listdir(MAIN_PATH):
        if date.split("_")[0].isdigit() == False:
            continue
        print(date)
        SIM_PATH = os.path.join(MAIN_PATH, date + '/')
        OBS_FILE = date 
        obs_runoff_df = data_5min_interpolate(load_runoff_obs_to_df(SIM_PATH + OBS_FILE))

        INP_FILE = r"raanana_28subcatchments.inp"
        inp = read_inp_file(SIM_PATH + INP_FILE) 
        subcatchment_d = dict(inp[sections.SUBCATCHMENTS])
        infiltration_d = dict(inp[sections.INFILTRATION])
        subareas_d = dict(inp[sections.SUBAREAS])
        timesereies_d = dict(inp[sections.TIMESERIES])

        for imp_factor, storage_factor, width_factor, n_factor, pct_zero_factor, cn_factor, pct_routed_factor, evap_factor in FACTOR_COMBINATION:
            update_imperviousness_with_factor(subcatchment_d, imp_factor)
            update_storage_with_factor(subareas_d, storage_factor)
            update_width_with_factor(subcatchment_d, width_factor, width_initial_values)
            update_n_with_factor(subareas_d, n_factor)
            update_pct_zero_with_factor(subareas_d, pct_zero_factor, pct_zero_initial_values)
            update_curve_num_with_factor(infiltration_d, cn_factor, curve_no_initial_values)
            update_pct_routed_with_factor(subareas_d, 'PERVIOUS', pct_routed_factor)
            inp['EVAPORATION']['CONSTANT'] = evap_factor

            file_name = 'calibration'
            inp.write_file(SIM_PATH + file_name + '.inp')
#             %timeit %memit swmm5_run(SIM_PATH + file_name + '.inp', progress_size=1)  ## Makes the cod slow

            swmm5_run(SIM_PATH + file_name + '.inp', progress_size=1)
            OUT_FILE = file_name + '.out'
            sim_df = load_sim_to_df(SIM_PATH+OUT_FILE)

            storm_df = pd.concat([obs_runoff_df, sim_df], axis=1)
            storm_df[storm_df.select_dtypes(np.float64).columns] = storm_df.select_dtypes(np.float64).astype(np.float16)
            storm_df[storm_df < 0] = 0
            storm_df = storm_df.fillna(0)

            if imp_factor == 1 and width_factor == 1 and storage_factor == 1 and n_factor == 1 and pct_zero_factor == 1 and cn_factor == 1:
                fig = observation_and_swmm_hydrograph(storm_df, date)
                fig.savefig(os.path.join(r'D:\Development\RESEARCH\Raanana\figures\cross_validation\hydrographs/', date + 'cross_val_28_catchments.png'), dpi=900)

            swmm_total_runoff, obs_total_runoff, swmm_max_runoff, obs_max_runoff, swmm_max_runoff_time = stat(storm_df)  
            
            imp_factor_l.append(imp_factor)
            width_factor_l.append(width_factor)
            storage_factor_l.append(storage_factor)
            n_factor_l.append(n_factor)
            pct_zero_factor_l.append(pct_zero_factor)
            cn_factor_l.append(cn_factor)
            pct_routed_factor_l.append(pct_routed_factor)
            evap_factor_l.append(evap_factor)
            
            swmm_total_runoff_l.append(swmm_total_runoff)
            swmm_max_runoff_l.append(swmm_max_runoff)
            swmm_max_runoff_time_l.append(swmm_max_runoff_time)
            swmm_runoff_l.append(list(storm_df['SWMM outflow [CMS]'].values))

            factor = pd.DataFrame(columns=pd.MultiIndex.from_product([['Factors'], ['Width', 'IMP', 'Storage', 'N', 'PCT_ZERO','CN', 'PCT_ROUTED', 'EVAP']]))
            cali_results_df = pd.DataFrame(columns=pd.MultiIndex.from_product([[date], ['Total Volume', 'Max Runoff', 'Max Runoff Time']]))
            cali_results_df = pd.concat([factor, cali_results_df], axis=0)

            cali_results_df[('Factors','IMP')] = imp_factor_l
            cali_results_df[('Factors','Width')] = width_factor_l
            cali_results_df[('Factors','Storage')] = storage_factor_l  
            cali_results_df[('Factors','N')] = n_factor_l
            cali_results_df[('Factors','PCT_ZERO')] = pct_zero_factor_l
            cali_results_df[('Factors','CN')] = cn_factor_l
            cali_results_df[('Factors','PCT_ROUTED')] = pct_routed_factor_l
            cali_results_df[('Factors','EVAP')] = evap_factor_l
                        
            cali_results_df[(date,'Total Volume')] = swmm_total_runoff_l
            cali_results_df[(date,'Max Runoff')] = swmm_max_runoff_l
            cali_results_df[(date,'Max Runoff Time')] = swmm_max_runoff_time_l  

            factor = pd.DataFrame(columns=pd.MultiIndex.from_product([['Factors'], ['Width','IMP','Storage','N', 'PCT_ZERO', 'CN', 'PCT_ROUTED', 'EVAP']]))
            new_columns = pd.MultiIndex.from_tuples([(date, str(index)) for index in storm_df['SWMM outflow [CMS]'].index])
            hydrograph_df = pd.DataFrame(data=swmm_runoff_l, columns=new_columns)
            factor[('Factors','IMP')] = imp_factor_l
            factor[('Factors','Width')] = width_factor_l
            factor[('Factors','Storage')] = storage_factor_l
            factor[('Factors','N')] = n_factor_l
            factor[('Factors','PCT_ZERO')] = pct_zero_factor_l
            factor[('Factors','CN')] = cn_factor_l
            factor[('Factors','PCT_ROUTED')] = pct_routed_factor_l
            factor[('Factors','EVAP')] = evap_factor_l
            
            hydrograph_df = pd.concat([factor, hydrograph_df], axis=1)

            inp = read_inp_file(SIM_PATH + INP_FILE)
            subareas_d = dict(inp[sections.SUBAREAS])
            infiltration_d = dict(inp[sections.INFILTRATION])
            subcatchment_d = dict(inp[sections.SUBCATCHMENTS])
            timesereies_d = dict(inp[sections.TIMESERIES])
    
        mi = pd.MultiIndex.from_tuples([(date, i) for i in storm_df.index], names=['Date', 'Time'])
        storm_df = pd.DataFrame(storm_df.values, index=mi, columns=storm_df.columns)
        obs_hydrograph_df = pd.concat([obs_hydrograph_df, storm_df[['OBS runoff [CMS]', 'rainfall [mm/h]']]], axis=0)

        cali_results_df_l += [cali_results_df]
        hydrograph_df_l += [hydrograph_df]
        
        imp_factor_l, width_factor_l, storage_factor_l = [], [], []
        pct_routed_factor_l, evap_factor_l = [], []
        pct_zero_factor_l, n_factor_l, cn_factor_l, swmm_runoff_l = [], [], [], []
        swmm_max_runoff_time_l, swmm_total_runoff_l, swmm_max_runoff_l = [], [], []


### Load Timeseries Data

In [12]:
# Specify the directory to load the pickle file from
directory = r"D:\Development\RESEARCH\Raanana\data\rain_radar\Basin_radar_overlap_pkl\Basin_radar_overlap_pkl.pkl"
# Combine directory and filename to get the full file path
filepath = os.path.join(directory)
# Load the pickle file
with open(filepath, "rb") as f:
    basins_rain_dfs_dict = pickle.load(f)

### Update swmm inpfile timeseries

In [13]:
# copyfrom_inp = read_inp_file(r'D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation_28subcatchments_with5min_interpolation_correction\2012_01_13\raanana_28subcatchments.inp')
# subcatch_tocopy = copyfrom_inp['SUBCATCHMENTS']                   
# jun_tocopy = copyfrom_inp['JUNCTIONS']                 
# con_tocopy = copyfrom_inp['CONDUITS']         
# xsections_tocopy = copyfrom_inp['XSECTIONS']         

In [14]:
MAIN_PATH = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/"
daily_eva = 1
INP_FILE = r"raanana_28subcatchments.inp"   ## This is the file name before manipulations
for date in os.listdir(MAIN_PATH):
    if date.split("_")[0].isdigit() == False:
        continue
#     print(date)
    SIM_PATH = os.path.join(MAIN_PATH, date + '/')
    inp = read_inp_file(SIM_PATH + INP_FILE)
    timesereies_d = dict(inp[sections.TIMESERIES])
    subareas_d = dict(inp[sections.SUBAREAS])
    basins_rain_df = basins_rain_dfs_dict[date]
    updated_timeseries_text = update_TimeSeriesData(timesereies_d, basins_rain_df)
    inp[sections.TIMESERIES] = updated_timeseries_text
#     update_pct_routed_with_factor(subareas_d, 'PERVIOUS', 20)
    inp['EVAPORATION']['CONSTANT'] = daily_eva
#     inp['SUBCATCHMENTS'] = subcatch_tocopy
#     inp['JUNCTIONS'] = jun_tocopy
#     inp['CONDUITS'] = con_tocopy
#     inp['XSECTIONS'] = xsections_tocopy
    ## Cunduit coreection:

    ## make sure that time serie data fit number of 'raingauges'
    if len(inp[sections.TIMESERIES].keys()) != len(inp[sections.RAINGAGES].keys()):
        print('ERROR')
    
    ## Edit the time series name and its corresponding rain gauge field so that they will be the same
    for basin_timeserie in range(len(list(inp[sections.TIMESERIES].keys()))):
#     print(basin_timeserie)
        basin_timeserie_name = inp[sections.TIMESERIES][list(inp[sections.TIMESERIES].keys())[basin_timeserie]]['name']
        inp[sections.RAINGAGES][list(inp[sections.RAINGAGES].keys())[basin_timeserie]]['timeseries'] = basin_timeserie_name
        
    file_name = 'raanana_28subcatchments'
    inp.write_file(SIM_PATH + file_name + '.inp')


## Calibration set-up

In [15]:
# this is working for sure!
imp_factors = np.arange(1, 1.2, 0.05)
storage_factors = np.arange(3, 8, 1)
width_factors = np.arange(0.4, 1.11, 0.3)
n_factors = np.arange(1, 1.1, 100)
pct_zero_factors = np.arange(1, 1.41, 0.7) 
cn_factors = np.arange(1, 1.1, 100)
pct_routed_factors = np.arange(30, 46, 5)
evap_factors = np.arange(3, 4.1, 1)

# Generate all combinations of factors
FACTOR_COMBINATION = np.array(list(itertools.product(imp_factors, storage_factors, width_factors,
                                                     n_factors, pct_zero_factors, cn_factors, pct_routed_factors,
                                                     evap_factors)))

# possiable due to memory issues: split the runs to few chunks by define num_runs>1.
## If you splie use 'Cross_Validation_concat_runs' code.
num_runs = 1

chunk_size = len(FACTOR_COMBINATION) // num_runs

# Divide the array into chunks
factor_combinations_runs_array = np.array(np.split(FACTOR_COMBINATION, num_runs))

# Now, factor_combinations_runs_array is a NumPy array containing 2 subarrays, each representing a portion of the combinations.
print(f'Chunk Size {chunk_size}')

Chunk Size 480


In [16]:
for num_run, combination in enumerate(factor_combinations_runs_array[0:], start=1):
    print(f'Processing Combination Run {num_run}/{num_runs}')
    run_calibration(MAIN_PATH, combination)
    
    # Merge the DataFrame lists by the factors value    
    merge_func = lambda left, right: pd.merge(left, right, how='left', on=[
        ('Factors','Width'), ('Factors','IMP'), ('Factors','Storage'),  
        ('Factors','N'), ('Factors','PCT_ZERO'), ('Factors','CN'),
        ('Factors','PCT_ROUTED'), ('Factors','EVAP')
    ])  # Include the new factors in the merge function

    cali_results_df = reduce(merge_func, cali_results_df_l)  # Apply the merge function iteratively to the list of DataFrames
    hydrograph_df = reduce(merge_func, hydrograph_df_l)  # Apply the merge function iteratively to the list of DataFrames

    cali_results_df[cali_results_df.select_dtypes(np.float64).columns] = cali_results_df.select_dtypes(np.float64).astype(np.float32)
    hydrograph_df[hydrograph_df.select_dtypes(np.float64).columns] = hydrograph_df.select_dtypes(np.float64).astype(np.float32)

    cali_swmm_hydrograph_df = hydrograph_df
    cali_results_df['Factors'] = cali_results_df['Factors'].round(2)
    
    ###---------------------------------- STEP 2 - Calibration Objective Functions Calculation----------------------###

    # Create an empty dictionary to store the DataFrames
    cross_validation_dict = {}
    cali_results_no_factors_df = cali_results_df.drop('Factors', level=0, axis=1)
    # Get the level 0 column names
    column_names = cali_results_no_factors_df.columns.get_level_values(0).unique()

    # Iterate over the level 0 column names
    for column_name in column_names:
        # Select all columns except the current column name
        cali_results_selected_columns = [(col_level_0, col_level_1) for col_level_0, col_level_1 in cali_results_no_factors_df if col_level_0 != column_name]
        cali_swmm_hydrograph_columns = [(col_level_0, col_level_1) for col_level_0, col_level_1 in cali_swmm_hydrograph_df if col_level_0 != column_name]
        obs_hydrograph_columns = [(col_level_0, col_level_1) for col_level_0, col_level_1 in obs_hydrograph_df.transpose() if col_level_0 != column_name]

        results_cali_df = cali_results_no_factors_df.loc[:, cali_results_selected_columns].copy()
        hydro_swmm_cali_df = cali_swmm_hydrograph_df.loc[:, cali_swmm_hydrograph_columns].copy()
        hydro_obs_df = obs_hydrograph_df.transpose().loc[:, obs_hydrograph_columns].copy()
        hydro_obs_df = hydro_obs_df.transpose()

        # Compute objective functions
        storm_obj_df, peak_obj_df, volume_obj_df = compute_objective_functions(hydro_obs_df, results_cali_df, hydro_swmm_cali_df)
 
        # Create a new DataFrame with the selected columns
        results_cali_df = pd.concat([cali_results_df[cali_results_df.columns.get_level_values(0)[:1]], results_cali_df, storm_obj_df, peak_obj_df, volume_obj_df], axis=1)

        # Generate the DataFrame name
        key_name = f'validation_{column_name}'

        # Store the DataFrame in the dictionary with the generated name
        cross_validation_dict[key_name] = [results_cali_df, hydro_swmm_cali_df, hydro_obs_df]
        
        # Define the path and filename
        path = r'D:\Development\RESEARCH\Raanana\SWMM/'
        ## for multiple runs:
        # filename = os.path.join(path, f'cross_validation_cali_dict_28catchments_part_{num_run}.pickle')
        ## for only ONE run:
        filename = os.path.join(path, f'cross_validation_cali_dict_28catchments.pickle')
    
        # Save the dictionary as a pickle file
        with open(filename, 'wb') as file:
            pickle.dump(cross_validation_dict, file)


Processing Combination Run 1/1
2012_01_13


swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2012_01_13/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2012_01_13/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2012_01_13/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2012_01_13/calibration.inp:   0%|      …

...

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2020_01_19/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2020_01_19/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2020_01_19/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2020_01_19/calibration.inp:   0%|      …

swmm5 D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation/2020_01_19/calibration.inp:   0%|      …